In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
import openai
import psycopg2
import time

In [2]:
load_dotenv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\.env")

True

In [69]:
conn = psycopg2.connect(
    host=os.environ.get("PGHOST"),
    port=os.environ.get("PGPORT"),
    user=os.environ.get("PGUSER"),
    password=os.environ.get("PGPASSWORD"),
    database=os.environ.get("PGDATABASE"),
)

In [7]:
def translate(text):
    response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo",
      messages=[
        {"role": "system", "content": "You are a translation engine that, in the contex of comex, can only translate text from english to spanish and cannot interpret it."},
        {"role": "user", "content": f"Translate:\n{text}"},
      ],
      temperature=0,
      n=1,
    )
    translation = response.choices[0].message.content.strip()
    if (translation[-1] == ".") and (text[-1] != "."):
      translation = translation[:-1]

    return translation

translate("Add Dialect")

'Agregar dialecto'

In [51]:
def load_data():
    if os.path.exists("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv"):
        data = pd.read_csv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv")
    else:
        with conn.cursor() as cur:
            cur.execute("SELECT dsmsgkey, dsenus, dsptbr, dseses FROM smart.i18n where dtdeleted is not null and dseses is null")
            data = pd.DataFrame(cur.fetchall(), columns=["dsmsgkey", "dsenus", "dsptbr", "dseses"])
    return data

In [9]:
def save_csv(data):
    data.to_csv("R:\\Richard\\src\\GitHub\\BCJTI\\translate-smart-gpt3\\translations.csv", index=False)

In [34]:
def save_data(data):
    save_csv(data)
    with conn.cursor() as cur:
        for _, row in data.iterrows():
            cur.execute(
                "UPDATE smart.i18n SET dseses = %s WHERE dsmsgkey = %s",
                (row["dseses"], row["dsmsgkey"]),
            )
        conn.commit()

In [52]:
data = load_data()
data

,dsmsgkey,dsenus,dsptbr,dseses
0,Caption.nmOrigin,Origin,Origem,None
1,Caption.nmAgent,Agent,Agente,None
2,Caption.nmCustomer,Customer,Cliente,None
3,Caption.nmCourier,Courier,Courier,None
4,Caption.nmCarrierDest,Carrier Destination,Armador Destino,None
...,...,...,...,...
1181,Label.financial-invoices-total,Invoice List,Lista de Invoices,None
1182,Label.nmAgentPayProc,Remittance Agent (Edited),Agente Remessa (Editado),None
1183,EditQuotation.button.documentGen,View document,Gerar Documento,None
1184,Label.trackingEmailDefaultSend,"By default, sending is done using the system e...","Por padrão, o envio é feito utilizando o email...",None


In [12]:
save_csv(data)

In [13]:
translate("Air")

'Aire'

In [65]:
for idx, row in data.iterrows():
    if pd.isna(row["dseses"]):
        try:
            translation = translate(row["dsenus"])
            data.loc[idx, "dseses"] = translation
            save_csv(data)
            print(f"Translated {row['dsmsgkey']}")
            time.sleep(3)
        except:
            print(f"Error translating {row['dsmsgkey']}")
            time.sleep(3)


Translated Caption.ownContract
Translated CommercialForm.label.quotation.obCustomerContact
Translated SalesChargeForm.label.flAccountable
Translated AwbSequenceForm.label.nbAwbSequence
Translated EditProcessCommodities.placeholder.commodity5
Translated EditOffice.button.negotiation
Translated CommercialForm.label.pricing.tpProfitShare


In [66]:
data["dseses"].isnull().sum()

0

In [67]:
save_csv(data)

In [70]:
save_data(data)

In [71]:
conn.close()
